In [0]:
# Notebook 03 — Intrinsic Compositional Descriptors
# Objective: Compute weighted-average, site-specific, delta, and structural descriptors
#            from composition table and atom_props_curated, and write ml_perovskites.X_intrinsic.
# Dependencies: ml_perovskites.master, ml_perovskites.composition, ml_perovskites.atom_props_curated
# Output: ml_perovskites.X_intrinsic (Delta table)
# Seed: 42 (fixed for reproducibility where applicable)
# Author: Bernardo Araldi da Silva | UFSC | 2026

# =============================================================================
# SECTION 0 — Install dependencies
# =============================================================================

# CELL: pip install (run alone before other cells)
# %pip install pymatgen --quiet

# =============================================================================
# SECTION 1 — Imports and configuration
# =============================================================================

import numpy as np
import pandas as pd
import warnings
from pyspark.sql import functions as F

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Oxygen ionic radius (Shannon, VI-coordinate, O2-)
R_OXYGEN_A = 1.40  # Angstrom — used in Goldschmidt tolerance factor

# Properties for which A-B delta descriptors are computed
DELTA_PROPERTIES = [
    "en_pauling",            # Pauling electronegativity (column name in atom_props)
    "atomic_radius",
    "ionic_radius_shannon_vi",
    "first_ion_en",
    "electron_affinity",
    "atomic_weight",
    "mendeleev_number",
]

print("Section 1 complete: imports and configuration loaded.")
print(f"Oxygen ionic radius (Shannon VI): {R_OXYGEN_A} Å")

# =============================================================================
# SECTION 2 — Load atom_props_curated with defensive header check
# =============================================================================

print("\n--- Loading atom_props_curated ---")

atom_props_spark = spark.table("ml_perovskites.atom_props_curated")
atom_props_raw = atom_props_spark.toPandas()

print(f"Raw shape: {atom_props_raw.shape}")
print(f"First 5 columns: {list(atom_props_raw.columns[:5])}")
print(atom_props_raw.head(2))

# Defensive header check: if columns look like _c0, _c1, ... use row 0 as header
if atom_props_raw.columns[0].startswith("_c"):
    print("\nWARNING: Detected _c0/_c1 column names — using row 0 as header.")
    atom_props_raw.columns = atom_props_raw.iloc[0].astype(str).str.strip()
    atom_props_raw = atom_props_raw.iloc[1:].reset_index(drop=True)
    print(f"Shape after header fix: {atom_props_raw.shape}")
    print(f"New first 5 columns: {list(atom_props_raw.columns[:5])}")
else:
    print("\nHeaders look correct — no fix needed.")

# Identify the element symbol column
# XenonPy typically uses 'symbol'; fall back to scanning for likely candidates
symbol_col = None
for candidate in ["symbol", "Symbol", "element", "Element", "atomic_symbol"]:
    if candidate in atom_props_raw.columns:
        symbol_col = candidate
        break

if symbol_col is None:
    # Heuristic: find column whose values match element symbols (1-2 chars, capitalised)
    import re
    for col in atom_props_raw.columns:
        sample = atom_props_raw[col].dropna().astype(str).head(10).tolist()
        if all(re.match(r"^[A-Z][a-z]?$", s.strip()) for s in sample if s.strip()):
            symbol_col = col
            print(f"Inferred symbol column by content inspection: '{symbol_col}'")
            break

if symbol_col is None:
    raise ValueError(
        "Could not identify element symbol column in atom_props_curated. "
        "Inspect atom_props_raw.columns and set symbol_col manually."
    )

print(f"\nElement symbol column: '{symbol_col}'")

# Clean up and convert numeric columns
atom_props_raw[symbol_col] = atom_props_raw[symbol_col].astype(str).str.strip()
numeric_cols = [c for c in atom_props_raw.columns if c != symbol_col]
for col in numeric_cols:
    atom_props_raw[col] = pd.to_numeric(atom_props_raw[col], errors="coerce")

# Drop rows with missing symbol
atom_props = atom_props_raw.dropna(subset=[symbol_col]).copy()
atom_props = atom_props.set_index(symbol_col)

print(f"\natom_props shape (indexed by symbol): {atom_props.shape}")
print(f"Number of elements: {len(atom_props)}")
print(f"Numeric columns available: {len(numeric_cols)}")
print(f"\nFirst 2 rows of atom_props:")
print(atom_props.head(2))

# Build fast lookup dict: {symbol: {col: value}}
props_dict = atom_props.to_dict(orient="index")

# Identify available property names for delta computation
available_delta_props = [p for p in DELTA_PROPERTIES if p in atom_props.columns]
missing_delta_props = [p for p in DELTA_PROPERTIES if p not in atom_props.columns]
print(f"\nRequested delta properties found    : {available_delta_props}")
print(f"Requested delta properties NOT found: {missing_delta_props}")

# Check for ionic_radius and atomic_radius columns (used in tolerance factor)
IONIC_RADIUS_COL = None
for candidate in ["ionic_radius_shannon_vi", "ionic_radius", "ionic_radius_vi"]:
    if candidate in atom_props.columns:
        IONIC_RADIUS_COL = candidate
        break

ATOMIC_RADIUS_COL = None
for candidate in ["atomic_radius", "atomic_radius_rahm", "atomic_radius_alvarez"]:
    if candidate in atom_props.columns:
        ATOMIC_RADIUS_COL = candidate
        break

print(f"\nIonic radius column  : {IONIC_RADIUS_COL}")
print(f"Atomic radius column : {ATOMIC_RADIUS_COL}")

# Check for valence electron columns
VALENCE_COL = None
for candidate in ["num_valance", "num_valence", "nvalence", "valence_electrons", "num_unfilled"]:
    if candidate in atom_props.columns:
        VALENCE_COL = candidate
        break
print(f"Valence electron column: {VALENCE_COL}")

# =============================================================================
# SECTION 3 — Load composition table (parse_ok == 1)
# =============================================================================

print("\n--- Loading composition table ---")

comp_spark = spark.table("ml_perovskites.composition")
comp_all = comp_spark.toPandas()
print(f"Total composition rows: {len(comp_all)}")

comp = comp_all[comp_all["parse_ok"] == 1].copy().reset_index(drop=True)
print(f"Rows with parse_ok == 1: {len(comp)}")
print(f"Columns: {list(comp.columns)}")
print(comp.head(2))

# Identify el_{X} columns
el_cols = [c for c in comp.columns if c.startswith("el_")]
elements_in_comp = [c[3:] for c in el_cols]  # strip 'el_' prefix to get symbol
print(f"\nNumber of el_{{X}} stoichiometry columns: {len(el_cols)}")
print(f"Sample elements: {elements_in_comp[:10]}")

# Validate that elements in comp are present in atom_props
missing_elements = [e for e in elements_in_comp if e not in props_dict]
if missing_elements:
    print(f"\nWARNING: {len(missing_elements)} elements in composition not found in atom_props: {missing_elements}")
else:
    print("\nAll elements in composition table are present in atom_props.")

# =============================================================================
# SECTION 4 — Helper functions for descriptor computation
# =============================================================================

def get_prop(symbol, prop_col):
    """Return property value for element symbol; NaN if not found."""
    row = props_dict.get(symbol)
    if row is None:
        return np.nan
    return row.get(prop_col, np.nan)


def weighted_average(elements_weights: dict, prop_col: str) -> float:
    """
    Compute stoichiometry-weighted average of a property across a set of elements.

    Parameters
    ----------
    elements_weights : dict {symbol: weight} (weights need not be normalised)
    prop_col         : column name in atom_props

    Returns
    -------
    float (NaN if all values missing or total weight == 0)
    """
    total_weight = 0.0
    weighted_sum = 0.0
    for sym, wt in elements_weights.items():
        if wt is None or np.isnan(float(wt)) or float(wt) == 0:
            continue
        val = get_prop(sym, prop_col)
        if val is not None and not np.isnan(float(val)):
            weighted_sum += float(wt) * float(val)
            total_weight += float(wt)
    if total_weight == 0:
        return np.nan
    return weighted_sum / total_weight


def parse_site_elements(site_str):
    """
    Parse comma-separated site element string into a list of symbols.
    Returns [] if NaN or empty.
    """
    if not isinstance(site_str, str) or site_str.strip() == "":
        return []
    return [s.strip() for s in site_str.split(",") if s.strip()]


def site_weights(row, site_elements):
    """
    Extract stoichiometric weights for a list of site elements from a composition row.

    Returns
    -------
    dict {symbol: stoichiometric_amount}
    """
    result = {}
    for sym in site_elements:
        col = f"el_{sym}"
        val = row.get(col, 0.0)
        try:
            val = float(val)
        except (TypeError, ValueError):
            val = 0.0
        if val > 0:
            result[sym] = val
    return result


print("Section 4 complete: helper functions defined.")

# =============================================================================
# SECTION 5 — Compute whole-formula weighted-average descriptors (wav_{property})
# =============================================================================

print("\n--- Computing whole-formula weighted-average descriptors ---")

numeric_prop_cols = list(atom_props.columns)  # all numeric columns in atom_props

rows_wav = []
for idx, row in comp.iterrows():
    record = {"source_row_id": row["source_row_id"]}

    # Build full-formula element-weight dict
    formula_weights = {}
    for col in el_cols:
        sym = col[3:]  # strip 'el_'
        val = row.get(col, 0.0)
        try:
            val = float(val)
        except (TypeError, ValueError):
            val = 0.0
        if val > 0 and sym in props_dict:
            formula_weights[sym] = val

    # Weighted averages for all numeric properties
    for prop in numeric_prop_cols:
        record[f"wav_{prop}"] = weighted_average(formula_weights, prop)

    rows_wav.append(record)

df_wav = pd.DataFrame(rows_wav)
print(f"Whole-formula WAV descriptors shape: {df_wav.shape}")
print(f"Sample columns: {list(df_wav.columns[:6])}")

# =============================================================================
# SECTION 6 — Compute A-site and B-site weighted-average descriptors
# =============================================================================

print("\n--- Computing A-site and B-site weighted-average descriptors ---")

rows_site = []
for idx, row in comp.iterrows():
    record = {"source_row_id": row["source_row_id"]}

    a_elements = parse_site_elements(row.get("A_site_elements", ""))
    b_elements = parse_site_elements(row.get("B_site_elements", ""))

    a_weights = site_weights(row, a_elements)
    b_weights = site_weights(row, b_elements)

    for prop in numeric_prop_cols:
        record[f"wav_A_{prop}"] = weighted_average(a_weights, prop) if a_weights else np.nan
        record[f"wav_B_{prop}"] = weighted_average(b_weights, prop) if b_weights else np.nan

    rows_site.append(record)

df_site = pd.DataFrame(rows_site)
print(f"Site-specific WAV descriptors shape: {df_site.shape}")

# =============================================================================
# SECTION 7 — Compute A-B delta descriptors
# =============================================================================

print("\n--- Computing A-B delta descriptors ---")

delta_records = []
for idx, row in df_site.iterrows():
    record = {"source_row_id": row["source_row_id"]}
    for prop in available_delta_props:
        a_val = row.get(f"wav_A_{prop}", np.nan)
        b_val = row.get(f"wav_B_{prop}", np.nan)
        if pd.notna(a_val) and pd.notna(b_val):
            record[f"delta_{prop}"] = float(a_val) - float(b_val)
        else:
            record[f"delta_{prop}"] = np.nan
    delta_records.append(record)

df_delta = pd.DataFrame(delta_records)
print(f"Delta descriptors shape: {df_delta.shape}")
print(f"Delta columns: {[c for c in df_delta.columns if c != 'source_row_id']}")

# =============================================================================
# SECTION 8 — Compute structural descriptors
# =============================================================================

print("\n--- Computing structural descriptors ---")


def get_site_ionic_radius(site_weights_dict):
    """
    Compute weighted-average ionic radius for a site.
    Uses IONIC_RADIUS_COL if available and non-NaN; falls back to ATOMIC_RADIUS_COL.
    Returns NaN if neither is available.
    """
    if not site_weights_dict:
        return np.nan

    # Try ionic radius first
    if IONIC_RADIUS_COL is not None:
        r = weighted_average(site_weights_dict, IONIC_RADIUS_COL)
        if pd.notna(r) and r > 0:
            return r

    # Fall back to atomic radius
    if ATOMIC_RADIUS_COL is not None:
        r = weighted_average(site_weights_dict, ATOMIC_RADIUS_COL)
        if pd.notna(r) and r > 0:
            return r

    return np.nan


struct_records = []
for idx, row in comp.iterrows():
    record = {"source_row_id": row["source_row_id"]}

    a_elements = parse_site_elements(row.get("A_site_elements", ""))
    b_elements = parse_site_elements(row.get("B_site_elements", ""))
    a_weights_dict = site_weights(row, a_elements)
    b_weights_dict = site_weights(row, b_elements)

    r_A = get_site_ionic_radius(a_weights_dict)
    r_B = get_site_ionic_radius(b_weights_dict)
    r_O = R_OXYGEN_A

    # --- Goldschmidt tolerance factor ---
    # t = (r_A + r_O) / (sqrt(2) * (r_B + r_O))
    try:
        if pd.notna(r_A) and pd.notna(r_B) and r_B > 0:
            goldschmidt_t = (r_A + r_O) / (np.sqrt(2) * (r_B + r_O))
        else:
            goldschmidt_t = np.nan
    except Exception:
        goldschmidt_t = np.nan
    record["goldschmidt_t"] = goldschmidt_t

    # --- Octahedral factor ---
    # mu = r_B / r_O
    try:
        if pd.notna(r_B) and r_B > 0:
            octahedral_mu = r_B / r_O
        else:
            octahedral_mu = np.nan
    except Exception:
        octahedral_mu = np.nan
    record["octahedral_mu"] = octahedral_mu

    # --- r_A and r_B (for reference / downstream use) ---
    record["ionic_radius_A"] = r_A
    record["ionic_radius_B"] = r_B

    # --- Number of valence electrons on B site ---
    if VALENCE_COL is not None and b_weights_dict:
        record["valence_electrons_B"] = weighted_average(b_weights_dict, VALENCE_COL)
    else:
        record["valence_electrons_B"] = np.nan

    # --- Electronegativity difference A-B (Pauling scale; redundant with delta_en_pauling but kept for clarity) ---
    if "en_pauling" in atom_props.columns:
        en_A = weighted_average(a_weights_dict, "en_pauling") if a_weights_dict else np.nan
        en_B = weighted_average(b_weights_dict, "en_pauling") if b_weights_dict else np.nan
        if pd.notna(en_A) and pd.notna(en_B):
            record["electronegativity_diff_AB"] = float(en_A) - float(en_B)
        else:
            record["electronegativity_diff_AB"] = np.nan
    else:
        record["electronegativity_diff_AB"] = np.nan

    struct_records.append(record)

df_struct = pd.DataFrame(struct_records)
print(f"Structural descriptors shape: {df_struct.shape}")
print(f"Structural columns: {[c for c in df_struct.columns if c != 'source_row_id']}")

# =============================================================================
# SECTION 9 — Merge all descriptor sets
# =============================================================================

print("\n--- Merging descriptor sets ---")

# Start with composition identifiers
id_cols = ["source_row_id", "base_compound", "family_proposed", "formula_reduced",
           "n_elements", "n_A_elements", "n_B_elements", "A_site_elements", "B_site_elements"]
id_cols = [c for c in id_cols if c in comp.columns]

df_out = comp[id_cols].copy()

# Merge whole-formula WAV
df_out = df_out.merge(df_wav, on="source_row_id", how="left")

# Merge site-specific WAV
df_out = df_out.merge(df_site, on="source_row_id", how="left")

# Merge delta
df_out = df_out.merge(df_delta, on="source_row_id", how="left")

# Merge structural
df_out = df_out.merge(df_struct, on="source_row_id", how="left")

print(f"\nFinal merged descriptor table shape: {df_out.shape}")
print(f"Total columns: {len(df_out.columns)}")

# =============================================================================
# SECTION 10 — Acceptance criteria and quality checks
# =============================================================================

print("\n" + "=" * 60)
print("ACCEPTANCE CRITERIA CHECKS")
print("=" * 60)

# (1) Row count
print(f"\n[CHECK 1] Row count: {len(df_out)} (expected 682)")
assert len(df_out) == 682, f"Row count mismatch: got {len(df_out)}, expected 682"
print("  PASS")

# (2) No NaN in wav_electronegativity and wav_atomic_weight
wav_en_col = "wav_electronegativity" if "wav_electronegativity" in df_out.columns else None
wav_aw_col = "wav_atomic_weight" if "wav_atomic_weight" in df_out.columns else None

for col in [wav_en_col, wav_aw_col]:
    if col is not None:
        n_nan = df_out[col].isna().sum()
        print(f"\n[CHECK 2] NaN count in {col}: {n_nan} (expected 0)")
        if n_nan > 0:
            print(f"  WARNING: {n_nan} NaN values found in {col} — inspect these rows.")
        else:
            print("  PASS")
    else:
        print(f"\n[CHECK 2] Column not found — skipping (column may have different name in atom_props).")

# (3) NaN rates for all descriptor columns
descriptor_cols = [c for c in df_out.columns if c not in id_cols]
nan_rates = (df_out[descriptor_cols].isna().sum() / len(df_out) * 100).sort_values(ascending=False)
print(f"\n[CHECK 3] NaN rates for descriptor columns (top 20 by NaN rate):")
print(nan_rates.head(20).to_string())
high_nan = nan_rates[nan_rates > 50]
if len(high_nan) > 0:
    print(f"\n  WARNING: {len(high_nan)} columns have >50% NaN:")
    print(high_nan.to_string())

# (4) Goldschmidt tolerance factor sanity check
gt_col = "goldschmidt_t"
if gt_col in df_out.columns:
    gt_valid = df_out[gt_col].dropna()
    print(f"\n[CHECK 4] goldschmidt_t statistics (expected range ~0.7–1.1 for perovskites):")
    print(f"  min  : {gt_valid.min():.4f}")
    print(f"  mean : {gt_valid.mean():.4f}")
    print(f"  max  : {gt_valid.max():.4f}")
    print(f"  count: {len(gt_valid)} / {len(df_out)}")
    outliers_low  = (gt_valid < 0.5).sum()
    outliers_high = (gt_valid > 1.2).sum()
    if outliers_low + outliers_high > 0:
        print(f"  WARNING: {outliers_low} rows below 0.5, {outliers_high} rows above 1.2 — check radii.")
    else:
        print("  PASS (all values within 0.5–1.2)")

# (5) source_row_id uniqueness
n_unique_ids = df_out["source_row_id"].nunique()
print(f"\n[CHECK 5] source_row_id unique values: {n_unique_ids} / {len(df_out)}")
if n_unique_ids != len(df_out):
    print("  WARNING: duplicate source_row_ids detected — investigate.")
else:
    print("  PASS")

print("\n" + "=" * 60)

# =============================================================================
# SECTION 11 — Write output to Delta table
# =============================================================================

print("\n--- Writing ml_perovskites.X_intrinsic as Delta table ---")

# Convert to Spark DataFrame
spark_out = spark.createDataFrame(df_out)

# Overwrite Delta table
(
    spark_out.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_perovskites.X_intrinsic")
)

# Verify
count_written = spark.table("ml_perovskites.X_intrinsic").count()
print(f"\nRows written to ml_perovskites.X_intrinsic: {count_written}")
assert count_written == 682, f"Written row count mismatch: {count_written}"
print("Delta table verified successfully.")

# Final summary
print("\n" + "=" * 60)
print("NOTEBOOK 03 COMPLETE")
print("=" * 60)
print(f"Output table  : ml_perovskites.X_intrinsic")
print(f"Rows          : {count_written}")
print(f"Total features: {len(descriptor_cols)}")
print(f"  - wav_*     : {len([c for c in descriptor_cols if c.startswith('wav_') and not c.startswith('wav_A_') and not c.startswith('wav_B_')])}")
print(f"  - wav_A_*   : {len([c for c in descriptor_cols if c.startswith('wav_A_')])}")
print(f"  - wav_B_*   : {len([c for c in descriptor_cols if c.startswith('wav_B_')])}")
print(f"  - delta_*   : {len([c for c in descriptor_cols if c.startswith('delta_')])}")
print(f"  - structural: {len([c for c in descriptor_cols if c in ['goldschmidt_t', 'octahedral_mu', 'ionic_radius_A', 'ionic_radius_B', 'valence_electrons_B', 'electronegativity_diff_AB']])}")
print(f"\nJoin key (NOT a feature): source_row_id")
print(f"Next step: Notebook 04 — Experimental feature encoding and full feature matrix assembly.")